In [2]:
import warnings

warnings.filterwarnings("ignore")

import torch

torch.set_grad_enabled(False)

from datasets import Dataset
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
)
from phaedra import split_sources
from phaedra.sources.pdf import PDF


In [4]:
text = PDF.extract_text("bitcoin.pdf")
splits = split_sources([text])


def generator():
    for i, split in enumerate(splits, start=1):
        yield {
            "title": f"Title {i}",
            "text": split,
        }


import csv
import json

with open("bitcoin.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerow(["title", "text"])
    for i, split in enumerate(splits, start=1):
        writer.writerow([f"Title {i}", split])


print(len(splits))

with open("bitcoin.json", "w") as f:
    json.dump(
        [
            {
                "title": f"Title {i}",
                "text": split,
            }
            for i, split in enumerate(splits, start=1)
        ],
        f,
    )


dataset = Dataset.from_generator(generator)


Using custom data configuration default-93e5d2164f450612
Found cached dataset generator (/home/alen/.cache/huggingface/datasets/generator/default-93e5d2164f450612/0.0.0)


27


In [ ]:
tokenizer = DPRContextEncoderTokenizer.from_pretrained(
    "facebook/dpr-ctx_encoder-multiset-base"
)
encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-multiset-base")


In [ ]:
dataset_with_embeddings = dataset.map(
    lambda sample: {
        "embeddings": encoder(**tokenizer(sample["text"], return_tensors="pt"))[0][
            0
        ].numpy()
    }
)
dataset_with_embeddings.add_faiss_index(column="embeddings")
dataset_with_embeddings.get_index("embeddings").save("bitcoin_dataset.faiss")
dataset_with_embeddings.drop_index("embeddings")
dataset_with_embeddings.save_to_disk("bitcoin_dataset")


In [3]:
# To load your own indexed dataset built with the datasets library that was saved on disk. More info in examples/rag/use_own_knowledge_dataset.py
from transformers import RagRetriever

dataset_path = "bitcoin_dataset"
index_path = "bitcoin_dataset.faiss"
retriever = RagRetriever.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base",
    index_name="custom",
    passages_path=dataset_path,
    index_path=index_path,
)


You are using a model of type dpr to instantiate a model of type rag. This is not supported for all configurations of models and can yield errors.


AssertionError: Config has to be initialized with question_encoder and generator config